[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/00_p05_shape_manipulation.ipynb)

# 🔵 Prerequisite 5: Shape Manipulation

Multi-head attention, GQA, KV cache, cross-attention, the GPT-2 block, RoPE, and ViT patch embedding all rely on the same few moves: **reshape → transpose → (compute) → transpose → reshape**.
In practice, most attention bugs are shape bugs.

### What you'll learn
1. `view` vs `reshape`, and why `.contiguous()` exists
2. `transpose` vs `permute`
3. `squeeze` / `unsqueeze` / `flatten`
4. `cat` vs `stack`
5. `repeat_interleave` vs `repeat`
6. Sliding windows with `F.pad` and `unfold`
7. `expand` vs `repeat`, `unflatten`, `movedim`, `roll`, and the `*_as` helpers
8. Many ways to do it: equivalent reshapes and their trade-offs

### Exercises
| # | Function | Shows up in |
|---|----------|-------------|
| 1 | `split_heads(x, num_heads)` | #06 MHA, #10 GQA, #13 GPT-2, #14 KV Cache, #23 Cross-Attn |
| 2 | `merge_heads(x)` | same as above (the inverse) |
| 3 | `repeat_kv(x, n_rep)` | #10 GQA |
| 4 | `interleave(a, b)` | #24 RoPE |
| 5 | `patchify(img, patch_size)` | #27 ViT Patch Embedding |
| 6 | `sliding_windows(x, k, stride, padding)` | #22 Conv2d |

### Rules
- **No Python loops.** Use only view/reshape/transpose/permute/cat/stack/repeat-style ops, plus `unfold` and `F.pad` (Exercise 6).
- Do **not** use `F.unfold` (the im2col function) or `F.conv2d`.
- Do **not** use `einops`.

### 📐 Notation

| Symbol | Meaning |
|:---:|---|
| `B` | batch size |
| `S` | sequence length |
| `D` | model dimension |
| `H` | number of attention heads (Exercises 1–3); **image height** in Exercises 5–6 and lesson §6 |
| `d` | dimension per head, `D // H` |
| `H_kv` | number of key/value heads (GQA), also written `Hkv` |
| `n_rep` | how many query heads share each key/value head |
| `C` | image channels |
| `W` | image width |
| `p` | patch size (`patchify`) |
| `N` | number of patches |
| `p_row` | row inside a patch |
| `p_col` | column inside a patch |
| `k` | window (kernel) size in `sliding_windows` |
| `H_out` | number of windows along the height |
| `W_out` | number of windows along the width |

### How to use this notebook
1. **Run the lesson cells** (sections 1, 2, …) from top to bottom. They are demos: change them and re-run.
2. **Do the exercises in order.** Each one has its own `✏️ YOUR IMPLEMENTATION HERE` cell, followed by a debug cell for checking shapes before you submit.
3. **Submit at any time** with the `check("p05_shape_manipulation")` cell at the bottom. It grades every exercise at once, and each test is named after its exercise, so a partial score shows which ones are left.
4. **Stuck?** `hint("p05_shape_manipulation")` gives one hint per exercise; `00_p05_shape_manipulation_solution.ipynb` has the reference code.
5. **Already comfortable with this skill?** Prerequisites are optional. Skip straight to the problems listed in **🔓 What this unlocks** at the bottom.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn.functional as F

## 1. `view` vs `reshape` and contiguity

A tensor is a flat block of memory plus **strides** that say how far to jump for each dimension.
- `view` only re-interprets the strides. It never copies data, so it **fails** if the memory layout doesn't allow the new shape.
- `transpose` / `permute` just swap strides. The result is usually **non-contiguous**.
- `reshape` = `view` if possible, otherwise copy. `.contiguous()` forces a copy into the standard layout.

In [ ]:
x = torch.arange(6).view(2, 3)
print(x, x.stride(), x.is_contiguous())

xt = x.transpose(0, 1)                       # (3, 2)
print(xt, xt.stride(), xt.is_contiguous())   # strides swapped, not contiguous

try:
    xt.view(6)
except RuntimeError as e:
    print("view failed:", str(e)[:60], "...")
print("reshape works:", xt.reshape(6))
print("contiguous().view works:", xt.contiguous().view(6))

> **Reading order matters.** `x.view(2, 3)` fills the new shape in **row-major** order (last dim varies fastest). Splitting a dimension `D` into `(H, d)` with `view(..., H, d)` therefore puts the first `d` features into head 0, the next `d` into head 1, and so on.

## 2. `transpose` vs `permute`

- `transpose(i, j)` swaps exactly two dims.
- `permute(...)` lists the new order of **all** dims.

In [ ]:
x = torch.randn(2, 3, 4, 5)
print(x.transpose(1, 2).shape)          # (2, 4, 3, 5)
print(x.permute(0, 2, 3, 1).shape)      # (2, 4, 5, 3)  e.g. NCHW -> NHWC
print(x.transpose(-2, -1).shape)        # (2, 3, 5, 4)  the K^T in attention

## 3. `squeeze`, `unsqueeze`, `flatten`

In [ ]:
x = torch.randn(3, 1, 4)
print(x.squeeze(1).shape)       # (3, 4): remove a size-1 dim
print(x.unsqueeze(0).shape)     # (1, 3, 1, 4): add one
print(x.flatten(1).shape)       # (3, 4): merge dims 1..end
print(torch.randn(2, 3, 4).flatten(-2).shape)  # (2, 12): merge the last two

## 4. `cat` vs `stack`

- `cat` joins along an **existing** dim. Shapes must match everywhere else.
- `stack` creates a **new** dim. All shapes must match exactly.

*#14 KV Cache appends new keys with `cat` along the sequence dim.*

In [ ]:
a, b = torch.zeros(2, 3), torch.ones(2, 3)
print(torch.cat([a, b], dim=0).shape)    # (4, 3)
print(torch.cat([a, b], dim=1).shape)    # (2, 6)
print(torch.stack([a, b], dim=0).shape)  # (2, 2, 3)
print(torch.stack([a, b], dim=-1).shape) # (2, 3, 2)

## 5. `repeat_interleave` vs `repeat`

Both copy data, but in a different **order**:

In [ ]:
x = torch.tensor([1, 2, 3])
print(x.repeat_interleave(2))   # [1, 1, 2, 2, 3, 3]: each element repeated in place
print(x.repeat(2))              # [1, 2, 3, 1, 2, 3]: the whole tensor tiled

## 6. Sliding windows: `F.pad` and `unfold`

`x.unfold(dim, size, step)` returns a **view** containing every window of length `size` along `dim`, taken every `step` elements. The windows become a **new last dim**.
Unfold dim 2 and then dim 3 of an image `(B, C, H, W)` to get every `k × k` patch: `(B, C, H_out, W_out, k, k)`.

The number of windows along a dim of size `H`, with padding `p` and stride `s`:

$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

`F.pad(x, (left, right, top, bottom))` pads the **last** dims first: the first pair pads the last dim, the second pair the one before it.

In [ ]:
x = torch.arange(16.).view(1, 1, 4, 4)
w = x.unfold(2, 2, 2).unfold(3, 2, 2)      # 2x2 windows, stride 2
print(w.shape)                             # (1, 1, 2, 2, 2, 2)
print(w[0, 0, 0, 1])                       # top-right window: [[2, 3], [6, 7]]
print(F.pad(x, (1, 1, 1, 1)).shape)        # (1, 1, 6, 6): one zero on every side
print(x.unfold(3, 3, 1).shape)             # 1-D windows along W: (1, 1, 4, 2, 3)

## 7. `expand` vs `repeat`, and other helpers

- **`x.expand(*sizes)`** broadcasts size-1 dims **without copying**: the new dims get stride `0`, so every "copy" reads the same memory. **`x.repeat(*reps)`** really copies. Use `expand` when you only read the result; never write into an expanded tensor.
- **`x.unflatten(dim, sizes)`** is the readable inverse of `flatten`: `x.unflatten(-1, (H, d))` splits the model dim into heads.
- **`x.movedim(src, dst)`** moves one dim to a new position; `swapaxes` / `transpose` swap two.
- **`torch.roll(x, shifts, dims)`** shifts elements cyclically (wrapping around the end).
- **`*_as` helpers** take the shape from another tensor: `view_as(y)`, `reshape_as(y)`, `expand_as(y)`.

In [ ]:
x = torch.arange(3.).view(3, 1)
e = x.expand(3, 4)
r = x.repeat(1, 4)
print(e.stride(), e.data_ptr() == x.data_ptr())   # (1, 0) True: no copy
print(r.stride(), r.data_ptr() == x.data_ptr())   # (4, 1) False: a real copy

h = torch.randn(2, 5, 12)
print(h.unflatten(-1, (3, 4)).shape)              # (2, 5, 3, 4): heads split off
print(torch.randn(2, 3, 4, 5).movedim(1, -1).shape)   # (2, 4, 5, 3): channels last
print(torch.roll(torch.arange(5), shifts=1))      # [4, 0, 1, 2, 3]
print(torch.zeros(2, 3).expand_as(torch.empty(4, 2, 3)).shape)

## 8. Many ways to do it

| Goal | Equivalent forms | Trade-off |
|------|------------------|-----------|
| Split heads | `x.view(B, S, H, d).transpose(1, 2)`, `x.unflatten(-1, (H, d)).transpose(1, 2)`, `x.reshape(B, S, H, d).permute(0, 2, 1, 3)` | `view` fails loudly on non-contiguous input; `reshape` silently copies; `unflatten` doesn't need `B` and `S` |
| Repeat KV heads (GQA) | `k.repeat_interleave(n, dim=1)`, `k[:, :, None].expand(B, Hkv, n, S, d).reshape(B, Hkv * n, S, d)` | the `expand` form is how many production implementations do it: no copy until `reshape` |
| Add a dim | `x.unsqueeze(1)`, `x[:, None]`, `x.view(B, 1, D)` | all views; `[:, None]` is the most compact inside expressions |
| Move channels last | `x.permute(0, 2, 3, 1)`, `x.movedim(1, -1)` | `movedim` states the intent and doesn't need every dim listed |

In [ ]:
B, Hkv, S, d, n = 2, 2, 5, 4, 3
k = torch.randn(B, Hkv, S, d)
a = k.repeat_interleave(n, dim=1)
b = k[:, :, None].expand(B, Hkv, n, S, d).reshape(B, Hkv * n, S, d)
print(torch.equal(a, b))

x = torch.randn(B, S, 12)
print(torch.equal(x.view(B, S, 3, 4).transpose(1, 2), x.unflatten(-1, (3, 4)).transpose(1, 2)))

---
# ✏️ Exercises

Use the shape conventions from the attention problems: `B` = batch, `S` = sequence length, `D` = model dim, `H` = number of heads, `d = D // H` = head dim.

### Exercise 1: `split_heads(x, num_heads)`

Split the model dimension into heads and move the head axis in front of the sequence axis.

- `x`: `(B, S, D)` with `D % num_heads == 0`
- returns: `(B, H, S, d)` where `out[b, h, s, :] = x[b, s, h*d:(h+1)*d]`

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def split_heads(x: torch.Tensor, num_heads: int) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.arange(2 * 3 * 8).float().view(2, 3, 8)
h = split_heads(x, num_heads=4)
print(h.shape)        # expect torch.Size([2, 4, 3, 2])
print(h[0, 1, 0])     # expect x[0, 0, 2:4] = tensor([2., 3.])

### Exercise 2: `merge_heads(x)`

The inverse of `split_heads`.

- `x`: `(B, H, S, d)`
- returns: `(B, S, H*d)`, so that `merge_heads(split_heads(x, H))` equals `x`

*Watch out: after a transpose, `view` will fail. Why?*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def merge_heads(x: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.randn(2, 5, 12)
roundtrip = merge_heads(split_heads(x, num_heads=3))
print(roundtrip.shape, torch.equal(roundtrip, x))   # expect torch.Size([2, 5, 12]) True

### Exercise 3: `repeat_kv(x, n_rep)`

In Grouped-Query Attention, several query heads share one key/value head. Expand the KV heads so the shapes match the queries.

- `x`: `(B, H_kv, S, d)`
- returns: `(B, H_kv * n_rep, S, d)` where heads are ordered `[kv0, kv0, ..., kv1, kv1, ...]`: each KV head is repeated `n_rep` times **consecutively**.

*Query head `h` must use KV head `h // n_rep`. Which of `repeat` / `repeat_interleave` gives that order?*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.arange(2).float().view(1, 2, 1, 1)   # two KV heads with values 0 and 1
print(repeat_kv(x, 3).flatten())               # expect tensor([0., 0., 0., 1., 1., 1.])

### Exercise 4: `interleave(a, b)`

Merge two tensors along the last dim, alternating elements: `[a0, b0, a1, b1, ...]`.

- `a`, `b`: `(..., N)` (same shape, any number of leading dims)
- returns: `(..., 2N)`

*RoPE rotates pairs `(x[2k], x[2k+1])` and then has to put the rotated halves back in interleaved order.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def interleave(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    pass  # Replace this

In [ ]:
a = torch.tensor([[1, 2, 3], [7, 8, 9]])
b = torch.tensor([[10, 20, 30], [70, 80, 90]])
print(interleave(a, b))
# expect [[ 1, 10,  2, 20,  3, 30],
#         [ 7, 70,  8, 80,  9, 90]]

### Exercise 5: `patchify(img, patch_size)`

Cut images into non-overlapping `p × p` patches and flatten each patch into a vector, as in ViT.

- `img`: `(B, C, H, W)`, where `H` and `W` are divisible by `p`
- returns: `(B, N, C*p*p)` where `N = (H/p) * (W/p)`
- **Patch order:** row-major over the patch grid (left→right, then top→bottom)
- **Within a patch:** flatten in `(C, p_row, p_col)` order, i.e. `out[b, n] == img[b, :, r*p:(r+1)*p, c*p:(c+1)*p].flatten()` for patch `n = r * (W/p) + c`

*Hint: `reshape` to 6 dims, `permute`, `reshape` back.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def patchify(img: torch.Tensor, patch_size: int) -> torch.Tensor:
    pass  # Replace this

In [ ]:
img = torch.arange(1 * 1 * 4 * 4).float().view(1, 1, 4, 4)
print(img[0, 0])
patches = patchify(img, 2)
print(patches.shape)   # expect torch.Size([1, 4, 4])
print(patches[0])
# expect [[ 0,  1,  4,  5],   top-left
#         [ 2,  3,  6,  7],   top-right
#         [ 8,  9, 12, 13],   bottom-left
#         [10, 11, 14, 15]]   bottom-right

### Exercise 6: `sliding_windows(x, k, stride=1, padding=0)`

Extract every `k × k` window of an image batch, as a convolution sees it.

- `x`: `(B, C, H, W)`
- zero-pad `padding` pixels on **every** side first, then take windows of size `k` every `stride` pixels
- returns: `(B, C, H_out, W_out, k, k)` with `H_out = (H + 2·padding − k) // stride + 1` (same for `W_out`)

*#22 Conv2d is exactly this followed by P6's `patch_conv`.*

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_windows(x: torch.Tensor, k: int, stride: int = 1, padding: int = 0) -> torch.Tensor:
    pass  # Replace this

In [ ]:
x = torch.arange(16.).view(1, 1, 4, 4)
win = sliding_windows(x, k=3, stride=1, padding=1)
print(win.shape)          # expect torch.Size([1, 1, 4, 4, 3, 3])
print(win[0, 0, 0, 0])    # expect [[0, 0, 0], [0, 0, 1], [0, 4, 5]]: top-left window, padded

<!-- Generated from scripts/prereqs/content/p05_shape_manipulation.py by `make prereqs`. Edit that file instead of this notebook; manual edits are overwritten. -->
## ✅ Submit

Run the judge on **all** exercises in this notebook. Each test is labelled with the exercise it checks.

In [ ]:
from torch_judge import check, hint
check("p05_shape_manipulation")

In [ ]:
# 💡 Stuck? Uncomment for a nudge
# hint("p05_shape_manipulation")

---
## 🔓 What this unlocks

With **Shape Manipulation** in your toolkit, you're ready for these problems:

| # | Problem | Difficulty | What you'll reuse | Links |
|:---:|---|:---:|---|---|
| #6 | Multi-Head Attention | 🔴 Hard | `split_heads` → attention → `merge_heads`, the core MHA pattern | [Open](06_multihead_attention.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/06_multihead_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/06_multihead_attention.ipynb" target="_blank">Colab</a> |
| #23 | Cross-Attention | 🔴 Hard | Same split/merge, with different sequence lengths for queries and keys | [Open](23_cross_attention.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/23_cross_attention.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/23_cross_attention.ipynb" target="_blank">Colab</a> |
| #10 | Grouped Query Attention | 🔴 Hard | `repeat_kv`: share each KV head across a group of query heads | [Open](10_gqa.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/10_gqa.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/10_gqa.ipynb" target="_blank">Colab</a> |
| #13 | GPT-2 Block | 🔴 Hard | Split/merge heads inside a full pre-norm transformer block | [Open](13_gpt2_block.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/13_gpt2_block.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/13_gpt2_block.ipynb" target="_blank">Colab</a> |
| #14 | KV Cache Attention | 🔴 Hard | Split/merge heads, plus `torch.cat` of past and new keys along the sequence dim | [Open](14_kv_cache.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/14_kv_cache.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/14_kv_cache.ipynb" target="_blank">Colab</a> |
| #24 | RoPE | 🔴 Hard | `interleave`: put rotated even/odd pairs back in order | [Open](24_rope.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/24_rope.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/24_rope.ipynb" target="_blank">Colab</a> |
| #22 | Conv2d | 🟡 Medium | `sliding_windows`: `F.pad` + `unfold`, the patch extraction inside Conv2d | [Open](22_conv2d.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/22_conv2d.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/22_conv2d.ipynb" target="_blank">Colab</a> |
| #27 | ViT Patch Embedding | 🟡 Medium | `patchify`: the reshape → permute → reshape at the heart of ViT | [Open](27_vit_patch.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/27_vit_patch.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/27_vit_patch.ipynb" target="_blank">Colab</a> |
| #28 | Mixture of Experts | 🔴 Hard | `reshape(-1, D)` to flatten `(B, S, D)` tokens for routing, then reshape back | [Open](28_moe.ipynb) · <a href="https://github.com/lalo/TorchCode/blob/master/templates/28_moe.ipynb" target="_blank">GitHub</a> · <a href="https://colab.research.google.com/github/lalo/TorchCode/blob/master/templates/28_moe.ipynb" target="_blank">Colab</a> |

🗺️ See the full recommended order in <a href="https://github.com/lalo/TorchCode/blob/master/LEARNING_PATH.md" target="_blank">LEARNING_PATH.md</a>.